# Task 2: Object Detection — All 4 Models Compared

Run the same 5 test images through Florence-2, Qwen2.5-VL-3B, Qwen2.5-VL-7B, and Phi-4-multimodal.
Compare detection quality, bounding box output, and speed.

Images resized to 500x500 for consistency.

In [ ]:
import torch, time, os, json, re
import cv2, numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches

IMG_SIZE = 500
COLORS = ['#00FF00','#FF0000','#0000FF','#FFFF00','#FF00FF','#00FFFF','#80FF00','#FF8000']

print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Load Test Images

In [ ]:
# Upload these 5 images to the compute instance first
IMAGE_NAMES = ['sample_dog.jpg', 'sample_dog_cat_house.jpg', 'sample_image.jpg', 'sample_room.jpg', 'sample_street.jpg']

images = {}
fig, axes = plt.subplots(1, 5, figsize=(25, 5))
for i, name in enumerate(IMAGE_NAMES):
    if os.path.exists(name):
        img = Image.open(name).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
        images[name] = img
        axes[i].imshow(img); axes[i].set_title(name.replace('sample_','').replace('.jpg',''), fontsize=10); axes[i].axis('off')
    else:
        print(f'Missing: {name}')
plt.suptitle(f'Test Images ({IMG_SIZE}x{IMG_SIZE})', fontsize=14)
plt.tight_layout(); plt.show()
print(f'Loaded {len(images)} images.')

## Visualization Helper

In [ ]:
def show_results(image, detections, title='', ax=None):
    """Draw bounding boxes on image. detections = [{'label':str, 'bbox':[x1,y1,x2,y2]}]"""
    if ax is None:
        fig, ax = plt.subplots(1, 1, figsize=(6, 6))
    ax.imshow(image)
    for i, d in enumerate(detections):
        b = d['bbox']; c = COLORS[i % len(COLORS)]
        ax.add_patch(patches.Rectangle((b[0],b[1]), b[2]-b[0], b[3]-b[1], lw=2, ec=c, fc='none'))
        ax.text(b[0], b[1]-3, d['label'], fontsize=8, fontweight='bold', color='white',
                bbox=dict(boxstyle='round,pad=0.2', facecolor=c, alpha=0.85))
    ax.set_title(f'{title} ({len(detections)} obj)', fontsize=10)
    ax.axis('off')

---
## 1. Florence-2-large (Native Object Detection)

In [ ]:
from transformers import AutoModelForCausalLM, AutoProcessor

f2_proc = AutoProcessor.from_pretrained('microsoft/Florence-2-large-ft', trust_remote_code=True)
f2_model = AutoModelForCausalLM.from_pretrained('microsoft/Florence-2-large-ft', torch_dtype=torch.float16, trust_remote_code=True).to('cuda')
print('Florence-2-large loaded.')

In [ ]:
f2_results = {}
fig, axes = plt.subplots(1, len(images), figsize=(5*len(images), 5))

for idx, (name, img) in enumerate(images.items()):
    inputs = f2_proc(text='<OD>', images=img, return_tensors='pt').to('cuda', torch.float16)
    t = time.time()
    ids = f2_model.generate(**inputs, max_new_tokens=1024, do_sample=False, num_beams=3)
    elapsed = time.time() - t
    text = f2_proc.batch_decode(ids, skip_special_tokens=False)[0]
    parsed = f2_proc.post_process_generation(text, task='<OD>', image_size=(IMG_SIZE, IMG_SIZE))
    
    # Convert Florence format to our standard format
    dets = []
    if '<OD>' in parsed:
        for label, bbox in zip(parsed['<OD>'].get('labels',[]), parsed['<OD>'].get('bboxes',[])):
            dets.append({'label': label, 'bbox': bbox})
    
    f2_results[name] = {'detections': dets, 'time': elapsed}
    show_results(img, dets, f'Florence-2 ({elapsed:.2f}s)', axes[idx])

plt.suptitle('Florence-2-large: Object Detection (<OD>)', fontsize=14, y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# Also run dense region captioning
fig, axes = plt.subplots(1, len(images), figsize=(5*len(images), 5))
for idx, (name, img) in enumerate(images.items()):
    inputs = f2_proc(text='<DENSE_REGION_CAPTION>', images=img, return_tensors='pt').to('cuda', torch.float16)
    ids = f2_model.generate(**inputs, max_new_tokens=1024, do_sample=False, num_beams=3)
    text = f2_proc.batch_decode(ids, skip_special_tokens=False)[0]
    parsed = f2_proc.post_process_generation(text, task='<DENSE_REGION_CAPTION>', image_size=(IMG_SIZE, IMG_SIZE))
    dets = []
    if '<DENSE_REGION_CAPTION>' in parsed:
        for label, bbox in zip(parsed['<DENSE_REGION_CAPTION>'].get('labels',[]), parsed['<DENSE_REGION_CAPTION>'].get('bboxes',[])):
            dets.append({'label': label, 'bbox': bbox})
    show_results(img, dets, 'Dense Caption', axes[idx])
plt.suptitle('Florence-2-large: Dense Region Captioning', fontsize=14, y=1.02)
plt.tight_layout(); plt.show()

del f2_model, f2_proc; torch.cuda.empty_cache(); print('Florence-2 cleared.')

## 2. Qwen2.5-VL-3B (Prompt-based Detection)

In [ ]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor as QwenProcessor

q3_proc = QwenProcessor.from_pretrained('Qwen/Qwen2.5-VL-3B-Instruct')
q3_model = Qwen2_5_VLForConditionalGeneration.from_pretrained('Qwen/Qwen2.5-VL-3B-Instruct', torch_dtype=torch.bfloat16, device_map='auto')
print('Qwen2.5-VL-3B loaded.')

In [ ]:
DET_PROMPT = f'This image is {IMG_SIZE}x{IMG_SIZE} pixels. Detect at most 4 objects. For each object return label and bounding box as pixel coordinates.\n\nReturn ONLY a JSON array like this (values are for reference only, use actual values from the image but keep the structure the same):\n[{{"label":"dog","bbox":[150,100,350,400]}},{{"label":"ball","bbox":[390,340,450,400]}}]'

def parse_vlm_response(raw):
    """Extract detections from VLM text response."""
    # Strip markdown code blocks
    raw = re.sub(r'```json\s*', '', raw)
    raw = re.sub(r'```\s*', '', raw)
    raw = raw.strip()
    
    def normalize_det(d):
        """Handle bbox or bbox_2d field names."""
        if not isinstance(d, dict) or 'label' not in d: return None
        bbox = d.get('bbox') or d.get('bbox_2d')
        if bbox and len(bbox) == 4: return {'label': d['label'], 'bbox': bbox}
        return None
    
    try:
        p = json.loads(raw)
        if isinstance(p, dict):
            for v in p.values():
                if isinstance(v, list): p = v; break
        if isinstance(p, list):
            return [x for x in [normalize_det(d) for d in p] if x][:4]
    except: pass
    m = re.search(r'\[.*\]', raw, re.DOTALL)
    if m:
        try:
            return [x for x in [normalize_det(d) for d in json.loads(m.group())] if x][:4]
        except: pass
    pairs = re.findall(r'"label"\s*:\s*"([^"]+)".*?"bbox(?:_2d)?"\s*:\s*\[([\d\s,.]+)\]', raw, re.DOTALL)
    if pairs:
        seen, dets = set(), []
        for label, bs in pairs:
            if label in seen: continue
            nums = [float(x.strip()) for x in bs.split(',') if x.strip()]
            if len(nums) == 4: seen.add(label); dets.append({'label': label, 'bbox': nums})
        return dets[:4]
    return []

def infer_qwen(model, proc, img, prompt):
    msgs = [{'role':'user','content':[{'type':'image','image':img},{'type':'text','text':prompt}]}]
    text = proc.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = proc(text=[text], images=[img], return_tensors='pt', padding=True).to(model.device)
    t = time.time()
    ids = model.generate(**inputs, max_new_tokens=512, temperature=0.2, do_sample=True)
    elapsed = time.time() - t
    out = proc.batch_decode(ids[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)[0]
    return out, elapsed

print('Detection prompt and helpers ready.')

In [ ]:
q3_results = {}
fig, axes = plt.subplots(1, len(images), figsize=(5*len(images), 5))

for idx, (name, img) in enumerate(images.items()):
    raw, elapsed = infer_qwen(q3_model, q3_proc, img, DET_PROMPT)
    dets = parse_vlm_response(raw)
    q3_results[name] = {'detections': dets, 'time': elapsed, 'raw': raw}
    show_results(img, dets, f'Qwen-3B ({elapsed:.2f}s)', axes[idx])
    print(f'{name}: {raw[:150]}')

plt.suptitle('Qwen2.5-VL-3B: Object Detection', fontsize=14, y=1.02)
plt.tight_layout(); plt.show()

del q3_model, q3_proc; torch.cuda.empty_cache(); print('Qwen-3B cleared.')

## 3. Qwen2.5-VL-7B

In [ ]:
q7_proc = QwenProcessor.from_pretrained('Qwen/Qwen2.5-VL-7B-Instruct')
q7_model = Qwen2_5_VLForConditionalGeneration.from_pretrained('Qwen/Qwen2.5-VL-7B-Instruct', torch_dtype=torch.bfloat16, device_map='auto')
print('Qwen2.5-VL-7B loaded.')

In [ ]:
q7_results = {}
fig, axes = plt.subplots(1, len(images), figsize=(5*len(images), 5))

for idx, (name, img) in enumerate(images.items()):
    raw, elapsed = infer_qwen(q7_model, q7_proc, img, DET_PROMPT)
    dets = parse_vlm_response(raw)
    q7_results[name] = {'detections': dets, 'time': elapsed, 'raw': raw}
    show_results(img, dets, f'Qwen-7B ({elapsed:.2f}s)', axes[idx])
    print(f'{name}: {raw[:150]}')

plt.suptitle('Qwen2.5-VL-7B: Object Detection', fontsize=14, y=1.02)
plt.tight_layout(); plt.show()

del q7_model, q7_proc; torch.cuda.empty_cache(); print('Qwen-7B cleared.')

## 4. Phi-4-multimodal

In [ ]:
phi_proc = AutoProcessor.from_pretrained('microsoft/Phi-4-multimodal-instruct', trust_remote_code=True)
phi_model = AutoModelForCausalLM.from_pretrained('microsoft/Phi-4-multimodal-instruct', torch_dtype=torch.bfloat16, trust_remote_code=True, device_map='auto', _attn_implementation='eager')
print('Phi-4-multimodal loaded.')

In [ ]:
phi_results = {}
fig, axes = plt.subplots(1, len(images), figsize=(5*len(images), 5))

for idx, (name, img) in enumerate(images.items()):
    full = f'<|user|>\n<|image_1|>\n{DET_PROMPT}<|end|>\n<|assistant|>\n'
    inputs = phi_proc(full, images=[img], return_tensors='pt')
    generate_kwargs = {k: v.to('cuda') for k, v in inputs.items() if v is not None}
    generate_kwargs['max_new_tokens'] = 512
    generate_kwargs['num_logits_to_keep'] = 1
    t = time.time()
    ids = phi_model.generate(**generate_kwargs)
    elapsed = time.time() - t
    out = phi_proc.batch_decode(ids, skip_special_tokens=True)[0]
    if '<|assistant|>' in out: out = out.split('<|assistant|>')[-1].strip()
    dets = parse_vlm_response(out)
    phi_results[name] = {'detections': dets, 'time': elapsed, 'raw': out}
    show_results(img, dets, f'Phi-4 ({elapsed:.2f}s)', axes[idx])
    print(f'{name}: {out[:150]}')

plt.suptitle('Phi-4-multimodal: Object Detection', fontsize=14, y=1.02)
plt.tight_layout(); plt.show()

del phi_model, phi_proc; torch.cuda.empty_cache(); print('Phi-4 cleared.')

---
## Comparison Summary

In [ ]:
print(f'{"Image":<25} {"Florence-2":<20} {"Qwen-3B":<20} {"Qwen-7B":<20} {"Phi-4":<20}')
print('=' * 105)

for name in IMAGE_NAMES:
    short = name.replace('sample_','').replace('.jpg','')
    f2 = f2_results.get(name, {})
    q3 = q3_results.get(name, {})
    q7 = q7_results.get(name, {})
    phi = phi_results.get(name, {})
    
    f2_info = f"{len(f2.get('detections',[]))} obj, {f2.get('time',0):.2f}s"
    q3_info = f"{len(q3.get('detections',[]))} obj, {q3.get('time',0):.2f}s"
    q7_info = f"{len(q7.get('detections',[]))} obj, {q7.get('time',0):.2f}s"
    phi_info = f"{len(phi.get('detections',[]))} obj, {phi.get('time',0):.2f}s"
    
    print(f'{short:<25} {f2_info:<20} {q3_info:<20} {q7_info:<20} {phi_info:<20}')

print()
print('Key observations:')
print('- Florence-2 uses native <OD> task — structured bbox output, no prompt engineering')
print('- VLMs (Qwen, Phi) use text prompts — output quality depends on prompt and model consistency')
print('- Compare: which model found the most objects? Which had valid bboxes? Which was fastest?')